# ECON6083: Machine Learning in Economics
## Lecture 8 Exercise: Instrumental Variables, JIVE, and DML-IV

**Coverage:** Lecture 8


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, LassoCV, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
TRUE_EFFECT = 1.0
results_summary = []


---

## Part 1: Classical JIVE (Jackknife IV)

When many instruments are weak, 2SLS overfits the first stage because it uses the same observations to estimate the first-stage coefficients and to predict $\hat{D}$. JIVE uses leave-one-out predictions to avoid this overfitting.

**True treatment effect**: $\theta_0 = 1.0$


In [ ]:
np.random.seed(42)
n = 300
p_instr = 60

ability = np.random.randn(n)
Z = np.random.randn(n, p_instr)

true_effects = np.random.uniform(0.03, 0.06, p_instr)
true_effects[:2] = [0.15, 0.12]
D = Z @ true_effects + 0.4 * ability + np.random.randn(n)
y = TRUE_EFFECT * D + 0.3 * ability + np.random.randn(n)

# 2SLS (in-sample first stage)
fs_ols = LinearRegression().fit(Z, D)
D_hat_ols = fs_ols.predict(Z)
theta_2sls = LinearRegression().fit(D_hat_ols.reshape(-1, 1), y).coef_[0]
r2_2sls = 1 - np.var(D - D_hat_ols) / np.var(D)

bias_2sls = abs(theta_2sls - TRUE_EFFECT)
print(f"2SLS:  estimate={theta_2sls:.3f}, bias={bias_2sls:.3f}, R²={r2_2sls:.3f}")


### Q1.1 Implement JIVE

Implement JIVE using the projection matrix formula (avoids $n$ separate regressions):

$$P = Z(Z'Z)^{-1}Z'$$

The JIVE estimator is:
$$\hat{\theta}_{\text{JIVE}} = \frac{D'(P - \text{diag}(h))Y}{D'(P - \text{diag}(h))D}$$

where $h = \text{diag}(P)$ is the leverage vector.

**Task**: Fill in the code below to compute the JIVE estimate and the honest leave-one-out $R^2$.


In [ ]:
# TODO: Compute projection matrix P and leverage h
ZTZ_inv = np.linalg.inv(Z.T @ Z)
P = Z @ ZTZ_inv @ Z.T
h = np.diag(P)

# TODO: Compute JIVE estimator
numerator = D @ (P - np.diag(h)) @ y
denominator = D @ (P - np.diag(h)) @ D
theta_jive = numerator / denominator

# TODO: Compute honest leave-one-out R²
D_hat_jive = (P - np.diag(h)) @ D / (1 - h)
r2_jive = 1 - np.var(D - D_hat_jive) / np.var(D)

bias_jive = abs(theta_jive - TRUE_EFFECT)
print(f"JIVE:  estimate={theta_jive:.3f}, bias={bias_jive:.3f}, R²={r2_jive:.3f}")
print(f"  -> 2SLS overfits: inflated R² by {r2_2sls - r2_jive:.3f}")
print(f"  -> JIVE bias reduction: {bias_2sls - bias_jive:.3f}")


### Q1.2 Interpretation

**Task**: Answer in the cell below.

1. Why does 2SLS produce an inflated first-stage $R^2$ compared to JIVE?
2. Which estimator is closer to the true effect of 1.0? Why?


*(Write your answers here)*

---

## Part 2: RJIVE — Regularized JIVE with Sparse vs Dense IV Structure

When $p > n$ or $p \approx n$, we need regularization in the first stage. The key question is: **Is the IV structure sparse or dense?**


In [ ]:
np.random.seed(42)
n = 800
p = 100
Z = np.random.randn(n, p)
ability = np.random.randn(n)

# Sparse structure: only 3 strong instruments
beta_sparse = np.zeros(p)
beta_sparse[:3] = [0.5, 0.4, 0.3]
D_sparse = Z @ beta_sparse + 0.4 * ability + np.random.randn(n)
y_sparse = TRUE_EFFECT * D_sparse + 0.4 * ability + np.random.randn(n)

# Dense structure: many weak instruments
beta_dense = np.random.uniform(0.03, 0.06, p)
D_dense = Z @ beta_dense + 0.3 * ability + np.random.randn(n)
y_dense = TRUE_EFFECT * D_dense + 0.3 * ability + np.random.randn(n)

print("Data generated: sparse (3 strong IVs) and dense (100 weak IVs)")



### Q2.1 K-Fold Jackknife Helper

**Task**: Implement `kfold_jackknife(Z, D, model, K=5)`.

Algorithm: split into K folds; for each fold, train on the rest and predict out-of-sample.


In [ ]:
def kfold_jackknife(Z, D, model, K=5):
    """K-fold Jackknife first stage."""
    n = len(D)
    D_hat = np.zeros(n)
    
    # TODO: Implement K-fold cross-fitting
    kf = KFold(n_splits=K, shuffle=True, random_state=42)
    for train_idx, test_idx in kf.split(Z):
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(Z[train_idx], D[train_idx])
        D_hat[test_idx] = model_clone.predict(Z[test_idx])
    
    return D_hat


### Q2.2 Sparse vs Dense in Practice

Now use your `kfold_jackknife` to compare Lasso-JIVE vs Ridge-JIVE under sparse and dense structures. Also compute the CV diagnostic.


In [ ]:
# Sparse: Lasso-JIVE (correct) vs Ridge-JIVE (wrong)
D_hat_lasso_s = kfold_jackknife(Z, D_sparse, LassoCV(cv=3, random_state=42), K=5)
theta_lasso_sparse = np.sum(y_sparse * D_hat_lasso_s) / np.sum(D_sparse * D_hat_lasso_s)

D_hat_ridge_s = kfold_jackknife(Z, D_sparse, RidgeCV(cv=3), K=5)
theta_ridge_sparse = np.sum(y_sparse * D_hat_ridge_s) / np.sum(D_sparse * D_hat_ridge_s)

D_hat_lasso_d = kfold_jackknife(Z, D_dense, LassoCV(cv=3, random_state=42, alphas=np.logspace(-4, -1, 20)), K=5)
theta_lasso_dense = np.sum(y_dense * D_hat_lasso_d) / np.sum(D_dense * D_hat_lasso_d)

D_hat_ridge_d = kfold_jackknife(Z, D_dense, RidgeCV(cv=3, alphas=np.logspace(-2, 2, 20)), K=5)
theta_ridge_dense = np.sum(y_dense * D_hat_ridge_d) / np.sum(D_dense * D_hat_ridge_d)


# CV diagnostic
r2_lasso_sparse = cross_val_score(LassoCV(cv=3), Z, D_sparse, cv=3, scoring="r2").mean()
r2_ridge_sparse = cross_val_score(RidgeCV(cv=3), Z, D_sparse, cv=3, scoring="r2").mean()
r2_lasso_dense = cross_val_score(LassoCV(cv=3), Z, D_dense, cv=3, scoring="r2").mean()
r2_ridge_dense = cross_val_score(RidgeCV(cv=3), Z, D_dense, cv=3, scoring="r2").mean()

print(f"Sparse: Lasso R²={r2_lasso_sparse:.3f}, Ridge R²={r2_ridge_sparse:.3f} -> Use Lasso")
print(f"Dense:  Lasso R²={r2_lasso_dense:.3f}, Ridge R²={r2_ridge_dense:.3f} -> Use Ridge")
print()
print(f"Sparse Lasso-JIVE: {theta_lasso_sparse:.3f} (bias: {abs(theta_lasso_sparse-TRUE_EFFECT):.3f})")
print(f"Sparse Ridge-JIVE: {theta_ridge_sparse:.3f} (bias: {abs(theta_ridge_sparse-TRUE_EFFECT):.3f})")
print(f"Dense Lasso-JIVE:  {theta_lasso_dense:.3f} (bias: {abs(theta_lasso_dense-TRUE_EFFECT):.3f})")
print(f"Dense Ridge-JIVE:  {theta_ridge_dense:.3f} (bias: {abs(theta_ridge_dense-TRUE_EFFECT):.3f})")



### Q2.3 Interpretation

**Task**: Answer briefly.

1. Why does Lasso-JIVE fail under dense IV structure?
2. How do you decide between Lasso and Ridge in practice?


*(Write your answers here)*

---

## Part 3: IV with Random Forests — Flexible Control via Sample Splitting

Treatment effects often vary by covariates. Classical 2SLS gives a single number (LATE), but we may want to better estimate the ATE by flexibly controlling for $X$.

**True ATE**: $\theta_0 = 1.0$


In [ ]:
np.random.seed(123)
n = 2000
X = np.random.randn(n, 5)
X1 = X[:, 0]
Z = np.random.randn(n)
ability = np.random.randn(n)

true_theta_x = TRUE_EFFECT - 0.5 * X1
D = 0.7 * Z + 0.4 * X1 + 0.4 * ability + np.random.randn(n)
y = true_theta_x * D + 0.4 * ability + 0.2 * X1 + np.random.randn(n)

# 2SLS
cov_yz = np.cov(y, Z)[0, 1]
cov_Dz = np.cov(D, Z)[0, 1]
theta_2sls_hte = cov_yz / cov_Dz
bias_2sls_hte = abs(theta_2sls_hte - TRUE_EFFECT)
print(f"2SLS:  {theta_2sls_hte:.3f}  (bias vs ATE: {bias_2sls_hte:.3f})")

# RF-IV: sample splitting
X_train, X_test, D_train, D_test, y_train, y_test, Z_train, Z_test = train_test_split(
    X, D, y, Z, test_size=0.3, random_state=42)

rf_y = RandomForestRegressor(n_estimators=50, max_depth=4, min_samples_leaf=30, random_state=42)
rf_D = RandomForestRegressor(n_estimators=50, max_depth=4, min_samples_leaf=30, random_state=42)
rf_y.fit(X_train, y_train)
rf_D.fit(X_train, D_train)
y_resid = y_test - rf_y.predict(X_test)
D_resid = D_test - rf_D.predict(X_test)
cov_yz_rf = np.cov(y_resid, Z_test)[0, 1]
cov_Dz_rf = np.cov(D_resid, Z_test)[0, 1]
theta_rf = cov_yz_rf / cov_Dz_rf if abs(cov_Dz_rf) > 0.01 else np.nan
bias_rf = abs(theta_rf - TRUE_EFFECT)
print(f"RF-IV: {theta_rf:.3f}  (bias vs ATE: {bias_rf:.3f})")


### Q3.1 Interpretation

**Task**: Answer briefly.

1. What is the purpose of using random forests to predict $y$ and $D$ before computing the IV estimate?
2. Why does sample splitting matter in RF-IV?


*(Write your answers here)*

---

## Part 4: DML-IV with Non-Linear Confounding

Now suppose confounders are high-dimensional and non-linear. Standard linear methods fail to capture confounding, leading to biased estimates even with an instrument.

**True treatment effect**: $\theta_0 = 1.0$


In [ ]:
np.random.seed(456)
n, p = 1000, 15
X_hd = np.random.randn(n, p)
Z_hd = np.random.randn(n)
ability_hd = np.random.randn(n)

X1, X2 = X_hd[:, 0], X_hd[:, 1]
confound = 3.0 * (X2 ** 2) - 2.0 * X1 * X2 + 1.5 * X1**2

D_hd = 0.8 * Z_hd + confound + 0.6 * ability_hd + np.random.randn(n)
y_hd = TRUE_EFFECT * D_hd + confound + 0.5 * ability_hd + np.random.randn(n)

# Naive OLS
naive_coef = LinearRegression().fit(
    np.hstack([D_hd.reshape(-1, 1), X_hd[:, :5]]), y_hd
).coef_[0]
bias_ols = abs(naive_coef - TRUE_EFFECT)
print(f"Naive OLS (linear):   {naive_coef:.3f}  (bias: {bias_ols:.3f})")

# 2SLS linear
fs_lin = LinearRegression().fit(np.hstack([Z_hd.reshape(-1, 1), X_hd[:, :5]]), D_hd)
D_hat_lin = fs_lin.predict(np.hstack([Z_hd.reshape(-1, 1), X_hd[:, :5]]))
theta_2sls_hd = LinearRegression().fit(D_hat_lin.reshape(-1, 1), y_hd).coef_[0]
bias_2sls_hd = abs(theta_2sls_hd - TRUE_EFFECT)
print(f"2SLS (linear X):      {theta_2sls_hd:.3f}  (bias: {bias_2sls_hd:.3f})")


### Q4.1 Implement DML-IV with Cross-Fitting

Use DML-IV to capture the non-linearity:
1. Expand $X$ to degree-2 polynomials
2. Use 3-fold cross-fitting to get out-of-sample residuals
3. Estimate $\hat{\theta} = \frac{\mathbb{E}[\tilde{Y} \cdot Z]}{\mathbb{E}[\tilde{D} \cdot Z]}$

**Task**: Fill in the code below.


In [ ]:
# TODO: Create polynomial features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_hd)

# TODO: Implement 3-fold cross-fitting
kf = KFold(n_splits=3, shuffle=True, random_state=42)
y_resid = np.zeros(n)
D_resid = np.zeros(n)

for train_idx, test_idx in kf.split(X_poly):
    # Train Ridge on other folds, predict residuals on test fold
    model_y = Ridge(alpha=1.0).fit(X_poly[train_idx], y_hd[train_idx])
    model_D = Ridge(alpha=1.0).fit(X_poly[train_idx], D_hd[train_idx])
    y_resid[test_idx] = y_hd[test_idx] - model_y.predict(X_poly[test_idx])
    D_resid[test_idx] = D_hd[test_idx] - model_D.predict(X_poly[test_idx])

# TODO: Compute DML-IV estimate
theta_dml = np.mean(y_resid * Z_hd) / np.mean(D_resid * Z_hd)

bias_dml = abs(theta_dml - TRUE_EFFECT)
print(f"DML-IV (polynomial):  {theta_dml:.3f}  (bias: {bias_dml:.3f})")


### Q4.2 Summary and Interpretation

**Task**: 
1. Fill in the summary table.
2. Answer: Why do linear methods fail here? What is the role of cross-fitting?


In [ ]:
# TODO: Append all results to results_summary
results_summary.append({"Part": "I. JIVE", "Scenario": "Many Weak IVs", "Method": "2SLS", "Estimate": theta_2sls, "True": TRUE_EFFECT, "Bias": bias_2sls})
results_summary.append({"Part": "I. JIVE", "Scenario": "Many Weak IVs", "Method": "JIVE", "Estimate": theta_jive, "True": TRUE_EFFECT, "Bias": bias_jive})
results_summary.append({"Part": "II. RJIVE", "Scenario": "Sparse IV", "Method": "Lasso-JIVE", "Estimate": theta_lasso_sparse, "True": TRUE_EFFECT, "Bias": abs(theta_lasso_sparse-TRUE_EFFECT)})
results_summary.append({"Part": "II. RJIVE", "Scenario": "Sparse IV", "Method": "Ridge-JIVE", "Estimate": theta_ridge_sparse, "True": TRUE_EFFECT, "Bias": abs(theta_ridge_sparse-TRUE_EFFECT)})
results_summary.append({"Part": "II. RJIVE", "Scenario": "Dense IV", "Method": "Lasso-JIVE", "Estimate": theta_lasso_dense, "True": TRUE_EFFECT, "Bias": abs(theta_lasso_dense-TRUE_EFFECT)})
results_summary.append({"Part": "II. RJIVE", "Scenario": "Dense IV", "Method": "Ridge-JIVE", "Estimate": theta_ridge_dense, "True": TRUE_EFFECT, "Bias": abs(theta_ridge_dense-TRUE_EFFECT)})
results_summary.append({"Part": "III. RF-IV", "Scenario": "Heterogeneous", "Method": "2SLS", "Estimate": theta_2sls_hte, "True": TRUE_EFFECT, "Bias": bias_2sls_hte})
results_summary.append({"Part": "III. RF-IV", "Scenario": "Heterogeneous", "Method": "RF-IV", "Estimate": theta_rf, "True": TRUE_EFFECT, "Bias": bias_rf})
results_summary.append({"Part": "IV. DML-IV", "Scenario": "Non-linear X", "Method": "Naive OLS", "Estimate": naive_coef, "True": TRUE_EFFECT, "Bias": bias_ols})
results_summary.append({"Part": "IV. DML-IV", "Scenario": "Non-linear X", "Method": "2SLS", "Estimate": theta_2sls_hd, "True": TRUE_EFFECT, "Bias": bias_2sls_hd})
results_summary.append({"Part": "IV. DML-IV", "Scenario": "Non-linear X", "Method": "DML-IV", "Estimate": theta_dml, "True": TRUE_EFFECT, "Bias": bias_dml})

df_summary = pd.DataFrame(results_summary)
print(df_summary.to_string(index=False))


*(Write your answers here)*

1. Why do linear methods fail here?
2. What is the role of cross-fitting?

*(Write your answers here)*

1. Why do linear methods fail here?
2. What is the role of cross-fitting?

---

## Submission Checklist

- [ ] All code cells run without error
- [ ] Key functions (JIVE, kfold_jackknife, DML-IV) are correctly implemented
- [ ] Written answers are complete and concise
- [ ] Summary table is displayed
